# Gán nhãn từ loại tiếng Việt bằng HMM

Notebook tái lập thí nghiệm train/dev của nhóm trên UD Vietnamese-VTB. Các cell chạy demo toy, tải và kiểm checksum hai split, thực hiện grid Bigram/Trigram và hiển thị kết quả dev cùng EDA. Phần đánh giá test được mô tả ở cuối.

Mở notebook cùng bản checkout dự án. Trong Colab, clone hoặc tải bản checkout đã tích hợp vào thư mục hiện hành rồi chạy từ đầu. Nếu đặt repo ở vị trí khác, đặt biến môi trường `NLP_REPO` trỏ đến thư mục gốc trước khi chạy. Dùng kernel Python có thể cài các thư viện trong `requirements.txt`.

In [ ]:
import csv
import os
from pathlib import Path
import subprocess
import sys

from IPython.display import Image, Markdown, display

cwd = Path.cwd()
candidates = (cwd, cwd.parent, cwd / "Vietnamese-POS-Tagging-HMM")
if os.getenv("NLP_REPO"):
    candidates = (Path(os.environ["NLP_REPO"]),)
repo = next(
    (path.resolve() for path in candidates if (path / "scripts" / "evaluate_dev.py").is_file()),
    None,
)
if repo is None:
    raise FileNotFoundError("Hãy đặt NLP_REPO trỏ đến bản checkout đã tích hợp của dự án.")

data_dir = Path(os.getenv("NLP_DATA_DIR", repo / "data/raw/ud-vtb-fa7130860b65")).resolve()
run_dir = Path(os.getenv("NLP_RUN_DIR", repo / ".venv/project-runs/notebook")).resolve()
predictions = run_dir / "predictions_dev.csv"
env = {**os.environ, "PYTHONUTF8": "1"}

def run(*arguments):
    result = subprocess.run(
        [sys.executable, *map(str, arguments)],
        cwd=repo,
        env=env,
        check=True,
        stdout=subprocess.PIPE,
        text=True,
        encoding="utf-8",
    )
    print(result.stdout, end="")

if os.getenv("NLP_SKIP_INSTALL") != "1":
    run("-m", "pip", "install", "-q", "-r", "requirements.txt")

## Demo trên câu toy

Hai câu train toy đủ để chạy B0, Bigram và Trigram. Bảng in nhãn vàng, dự đoán và trạng thái OOV của câu giữ lại. Dùng [ví dụ giải tay](../docs/TV2_GIAI_TAY_VITERBI.md) để kiểm tra backpointer và hiểu cách tính điểm đường đi.

In [ ]:
run("-m", "scripts.demo_tv2")

## Dữ liệu và tìm cấu hình trên dev

Hai split được ghim theo cùng một commit corpus và kiểm SHA-256 mỗi lần chạy. HMM học từ train, sau đó chọn cấu hình bằng dev. B0 và hai mô hình cố định α=0,1, cutoff=1 làm mốc so sánh. Bảng dự đoán từng token được giữ local trong thư mục Git bỏ qua.

In [ ]:
run(
    "-m", "scripts.prepare_tv1", "--download", "--data-dir", data_dir,
    "--tables-dir", run_dir / "eda", "--figures-dir", run_dir / "figures",
)
run(
    "-m", "scripts.evaluate_dev", "--data", data_dir, "--output", run_dir / "dev",
    "--predictions", predictions,
)
run(
    "-m", "scripts.prepare_tv1", "--data-dir", data_dir, "--dev-predictions", predictions,
    "--tables-dir", run_dir / "eda", "--figures-dir", run_dir / "figures",
)

In [ ]:
with (run_dir / "dev" / "metrics_dev.csv").open(encoding="utf-8", newline="") as stream:
    metrics = list(csv.DictReader(stream))

lines = [
    "| Mô hình | Alpha | Cutoff | Accuracy | Macro-F1 | OOV accuracy | No-PUNCT accuracy |",
    "| --- | ---: | ---: | ---: | ---: | ---: | ---: |",
]
for row in metrics:
    accuracy_columns = ("token_accuracy", "macro_f1", "oov_accuracy", "accuracy_no_punct")
    scores = [f"{float(row[key]):.2%}" for key in accuracy_columns]
    values = [row["model"], row["alpha"], row["cutoff"], *scores]
    lines.append("| " + " | ".join(values) + " |")

display(Markdown("\n".join(lines)))
for filename in ("sentence_lengths.png", "upos_distribution.png", "oov_by_upos.png"):
    display(Image(filename=run_dir / "figures" / filename, width=900))

## Đánh giá cuối sau khi nhóm khóa cấu hình

Đối chiếu cấu hình dev và hash corpus/code với [biên bản](../docs/pre_register_test.md). Khi cả nhóm xác nhận, dùng hướng dẫn trong README và CLI của script final để đóng băng candidate đã duyệt và chạy test một lần trên đúng corpus. Notebook này chạy train/dev; đầu ra theo token của test thuộc thư mục Git bỏ qua. Nguồn đối chiếu dữ liệu nằm trong [tài liệu TV1](../docs/TV1_DATA.md).

```sh
python -m scripts.final_experiment --help
python -m scripts.final_experiment run --help
```